In [2]:
from IPython.display import HTML
HTML("<style>.container{width:100%!important;margin:auto}div.cell,div.input_area{padding:2px;margin:0}div.output_wrapper{padding:0;margin:0}</style>")

In [7]:
import copy
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_fscore_support

In [21]:
# --- 1. DATA PREPARATION ---
data = load_iris()

X_raw = torch.tensor(data.data, dtype=torch.float32)
Y_raw = torch.tensor(data.target, dtype=torch.long)
print(X_raw.shape, Y_raw.shape)
print(np.unique(Y_raw))

X_train, X_test, Y_train, Y_test = train_test_split(X_raw, Y_raw, test_size=0.2, random_state=42)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.1, random_state=42)
print(X_train.shape, Y_train.shape)
print(X_train.shape, X_val.shape, X_test.shape)

scalar = StandardScaler()
X_train = torch.tensor(scalar.fit_transform(X_train), dtype=torch.float32)
X_val = torch.tensor(scalar.transform(X_val), dtype=torch.float32)
X_test = torch.tensor(scalar.transform(X_test), dtype=torch.float32)




torch.Size([150, 4]) torch.Size([150])
[0 1 2]
torch.Size([108, 4]) torch.Size([108])
torch.Size([108, 4]) torch.Size([12, 4]) torch.Size([30, 4])


In [22]:
# --- 2. CREATE PYTORCH DATASETS & DATALOADERS ---
train_dataset = TensorDataset(X_train, Y_train)
val_dataset = TensorDataset(X_val, Y_val)
test_dataset = TensorDataset(X_test, Y_test)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)




In [23]:
# --- 3. DEFINE THE MLP ---
class simpleMLP(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(simpleMLP, self).__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.hidden(x))
        x = self.output(x)
        return x


In [39]:
# --- 4. TRAINING SETUP ---
model = simpleMLP(input_dim=X_train.shape[1], hidden_dim=4, output_dim=len(np.unique(Y_raw)))
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01)

print("training started")

max_epochs = 100
for epoch in range(1, max_epochs+1):
    model.train()
    train_loss = 0
    accuracy = 0

    for batch_X, batch_Y in train_loader:
        optimizer.zero_grad()
        pred = model(batch_X)
        loss = criterion(pred, batch_Y)
        loss.backward()
        optimizer.step()

        train_loss = train_loss+loss.item()*batch_X.size(0)
        train_pred = pred.argmax(dim=1)
        accuracy = accuracy + (train_pred==batch_Y).sum().item()

    avg_loss = (train_loss/len(train_loader.dataset)) * 100
    avg_acc = (accuracy/len(train_loader.dataset))*100

    #----------------------Validation---------------------#
    model.eval()
    val_loss = 0
    val_acc = 0

    with torch.no_grad():
        for batch_X, batch_Y in val_loader:
            pred = model(batch_X)
            loss = criterion(pred, batch_Y)

            val_loss = val_loss+loss.item()*batch_X.size(0)
            val_pred = pred.argmax(dim=1)
            val_acc = val_acc + (val_pred==batch_Y).sum().item()
    
        avg_val_loss = (val_loss/len(val_loader.dataset)) * 100
        avg_val_acc = (val_acc/len(val_loader.dataset))*100


    print(f"Epoch {epoch}----> Loss: {avg_loss} & ------> Accuracy: {avg_acc} || Epoch {epoch}----> Loss: {avg_val_loss} & ------> Accuracy: {avg_val_acc}")


training started
Epoch 1----> Loss: 113.62565226025052 & ------> Accuracy: 33.33333333333333 || Epoch 1----> Loss: 104.1406512260437 & ------> Accuracy: 41.66666666666667
Epoch 2----> Loss: 110.40592016997162 & ------> Accuracy: 33.33333333333333 || Epoch 2----> Loss: 101.98725461959839 & ------> Accuracy: 41.66666666666667
Epoch 3----> Loss: 107.2900926625287 & ------> Accuracy: 33.33333333333333 || Epoch 3----> Loss: 99.96090531349182 & ------> Accuracy: 41.66666666666667
Epoch 4----> Loss: 104.38025659985013 & ------> Accuracy: 33.33333333333333 || Epoch 4----> Loss: 98.03141951560974 & ------> Accuracy: 41.66666666666667
Epoch 5----> Loss: 101.57839148132891 & ------> Accuracy: 33.33333333333333 || Epoch 5----> Loss: 96.15716934204102 & ------> Accuracy: 41.66666666666667
Epoch 6----> Loss: 98.85104144061053 & ------> Accuracy: 33.33333333333333 || Epoch 6----> Loss: 94.34424042701721 & ------> Accuracy: 41.66666666666667
Epoch 7----> Loss: 96.26927353717663 & ------> Accuracy: 37.

In [34]:
# --- 5. EVALUATION ---

model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for batch_X, batch_Y in test_loader:
        pred = model(batch_X)
        test_pred = pred.argmax(dim=1)
        all_preds.extend(test_pred.cpu().numpy())
        all_targets.extend(batch_Y.cpu().numpy())

target_names = data.target_names
print(classification_report(all_targets, all_preds, target_names=target_names))




              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.33      0.50         9
   virginica       0.65      1.00      0.79        11

    accuracy                           0.80        30
   macro avg       0.88      0.78      0.76        30
weighted avg       0.87      0.80      0.77        30



In [ ]:
# D. Early Stopping & Checkpointing Setup (Copy 4)



In [ ]:
# --- 5. EVALUATION --- (Copy 5)

In [ ]:
# --- Feature Importance via Input Gradient Analysis ---

In [ ]:
# Train on some other standard datasets
from sklearn.datasets import load_wine, load_digits, load_breast_cancer
import numpy as np
# Copy 1, 2, 3, D, 5